### Aufgabe 1 —Erstbesichtigung

Lade die CSV und prüfe: Wie viele Zeilen? Wie viele Spalten? Welche Typen?
- 508 Zeilen, 13 Spalten, Strings, floats, integers

df.describe() —welche Werte sind unplausibel? (Tipp: 1–2 Werte sind unmöglich.)
- Admission day 0, falls hier nicht zero-based indexing für Wochentage genutzt wird.
- Körpertemperatur von 35.0 könnte etwas sehr niedrig sein
- Maximaler Laktatwert von 8 ist um multiples höher als ein Normwert.

Schreibe 2–3 Beobachtungen auf —sie führen zu den Aufgaben 2–4.
- Es geht offenbar darum, ob bei Patienten eine Sepsis vorliegt, anhand der Blutwerte und allgemeiner Gesundheitsparameter.
- Admission day scheint sich auf die Tage eines Monats zu beziehen (0-30)

Notiert konkret: Zeilen ×Spalten, Datentypen (int/float/object), 1–2 verdächtige Werte —führt zu A2–A4.

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("uebung_w2_ehr_daten.csv")
print(df.shape) # Zeilen, Spalten?
df.info() # Typen + Non-Null-Zählungen
df.head()

(508, 13)
<class 'pandas.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   patient_id     508 non-null    str    
 1   age            508 non-null    int64  
 2   sex            508 non-null    str    
 3   bmi            508 non-null    float64
 4   systolic       508 non-null    float64
 5   heart_rate     508 non-null    float64
 6   temp           495 non-null    float64
 7   wbc            508 non-null    float64
 8   lactate        438 non-null    float64
 9   creatinine     458 non-null    float64
 10  glucose        508 non-null    float64
 11  admission_day  508 non-null    int64  
 12  sepsis         508 non-null    int64  
dtypes: float64(8), int64(3), str(2)
memory usage: 51.7 KB


,patient_id,age,sex,bmi,systolic,heart_rate,temp,wbc,lactate,creatinine,glucose,admission_day,sepsis
0,P0080,53,M,29.3,138.0,80.0,37.4,8.6,NaN,0.93,109.0,27,0
1,P0317,59,W,25.7,137.0,86.0,36.3,15.2,2.84,0.95,93.0,27,0
2,P0486,55,W,15.7,166.0,89.0,36.5,8.9,3.92,1.22,60.0,23,0
3,P0397,70,M,18.2,99.0,90.0,37.0,7.3,1.30,1.17,116.0,18,0
4,P0168,64,W,27.6,107.0,89.0,36.5,5.6,2.13,0.78,103.0,3,0


In [4]:
df.describe() # Min/Max/Mittel — was ist unplausibel?

,age,bmi,systolic,heart_rate,temp,wbc,lactate,creatinine,glucose,admission_day,sepsis
count,508.000000,508.000000,508.000000,508.000000,495.000000,508.000000,438.000000,458.000000,508.000000,508.000000,508.000000
mean,57.389764,27.080315,132.183071,80.366142,37.108283,7.757480,2.083950,0.962598,104.251969,15.074803,0.029528
std,15.179194,5.083415,17.973791,16.374795,0.869054,2.578205,1.049602,0.286517,23.978284,8.874887,0.169447
min,18.000000,15.000000,90.000000,40.000000,35.000000,2.000000,0.560000,0.400000,60.000000,0.000000,0.000000
25%,47.000000,23.700000,121.000000,70.000000,36.500000,6.000000,1.342500,0.760000,87.000000,8.000000,0.000000
50%,58.000000,27.100000,132.000000,81.000000,37.100000,7.800000,1.850000,0.945000,104.500000,15.000000,0.000000
75%,67.000000,30.425000,144.250000,90.000000,37.700000,9.400000,2.497500,1.127500,120.000000,23.000000,0.000000
max,95.000000,41.000000,187.000000,220.000000,40.000000,16.900000,8.000000,1.990000,176.000000,30.000000,1.000000


### Aufgabe 2 —Fehlende Werte

Wie viele fehlende Werte pro Spalte (absolut + %)? Welche Spalte ist am stärksten betroffen?
- Spalten Laktat und Kreatinin haben rund 10% fehlende Werte.

Welche Strategie passt (löschen / imputieren / Indikator-Spalte) —und WARUM? Begründet eure Wahl.
- Wir haben mehr gesunde als kranke, daher kann man annehmen dass sich fehlende Werte um den Median bewegen, solange man dem Modell mitgibt dass der echte Wert fehlt.

Implementiere für „creatinine“: Median-Imputation + Indikator-Spalte für fehlende Werte (Reihenfolge beachten!).
Tipp: absolute Zahlen (.isna().sum()) UND Anteile (.mean()) nebeneinander —kleine Spalten fallen in % auf.

In [5]:
df.isna().mean().sort_values(ascending=False) # fehlende Werte pro Spalte in % sortiert

lactate          0.137795
creatinine       0.098425
temp             0.025591
patient_id       0.000000
age              0.000000
sex              0.000000
bmi              0.000000
systolic         0.000000
heart_rate       0.000000
wbc              0.000000
glucose          0.000000
admission_day    0.000000
sepsis           0.000000
dtype: float64

In [6]:
df["creatinine_missing"] = df["creatinine"].isna().astype(int) # SPALTE zuerst!
df["creatinine"] = df["creatinine"].fillna(df["creatinine"].median())

### Aufgabe 3 —Duplikate & EMPI

Schritt 1: Wie viele EXAKTE Duplikat-Zeilen? Entferne sie.
- 10 Exakte Duplikate

Schritt 2 (EMPI): Welche Patienten (patient_id) erscheinen zweimal mit widersprüchlichen Daten? (Tipp: duplicated('patient_id').)
- 3 mal selbe Patienten-ID mit unterschiedlichen Aufnahmedaten und Blutwerten

Führe die Konflikt-Zeilen zu EINER Zeile zusammen (z. B. Mittelwerte der Labore, jüngstes Aufnahme-Datum).
Nach dem Merge: df.shape prüfen (3 Zeilen weniger) —und: keine patient_id mehr doppelt (duplicated-Check leer).

In [7]:
dupes = df[df.duplicated(keep=False)] 
print(len(dupes))

df = df.drop_duplicates().reset_index(drop=True) # Schritt 1

10


In [8]:
conf = df[df.duplicated("patient_id", keep=False)].sort_values("patient_id")
print(conf) # Schritt 2: Konflikte ansehen

    patient_id  age sex   bmi  systolic  heart_rate  temp   wbc  lactate  \
158      P0011   72   W  27.6     148.0        72.0  36.6   9.5     1.64   
249      P0011   72   W  27.7     148.0        72.0  36.6   9.5     1.64   
37       P0047   71   M  23.2     171.0        82.0  36.9  10.2      NaN   
77       P0047   71   M  24.4     171.0        82.0  36.9  10.2      NaN   
270      P0123   61   M  34.4     140.0        86.0  35.7   8.1     2.44   
494      P0123   61   M  36.7     140.0        86.0  35.7   8.1     2.44   

     creatinine  glucose  admission_day  sepsis  creatinine_missing  
158        0.85    116.0              8       0                   0  
249        0.76    111.0              2       0                   0  
37         0.88    119.0             26       0                   0  
77         0.73    105.0             20       0                   0  
270        1.12    107.0             19       0                   0  
494        1.39    109.0             21       0

In [9]:
# Konflikte zusammenführen: Mittelwerte der Labore, jüngstes Aufnahme-Datum

# Aggregationsregeln für jede Spalte definieren
agg_dict = {
    'age': 'first',                 # Konstanter Wert, behalte den ersten
    'sex': 'first',                 # Konstanter Wert, behalte den ersten
    'bmi': 'mean',                  # Mittelwert
    'systolic': 'mean',             # Mittelwert
    'heart_rate': 'mean',           # Mittelwert
    'temp': 'mean',                 # Mittelwert
    'wbc': 'mean',                  # Mittelwert
    'lactate': 'mean',              # Mittelwert
    'creatinine': 'mean',           # Mittelwert
    'glucose': 'mean',              # Mittelwert
    'admission_day': 'min',         # frühestes Datum
    'sepsis': 'first',              # Konstanter Wert (oder max, falls 1 Vorrang vor 0 haben soll)
    'creatinine_missing': 'first'   # Konstanter Wert
}

df = df.groupby('patient_id', as_index=False).agg(agg_dict)

print(df.shape)

(500, 14)


### Aufgabe 4 —Ungleichgewicht & Metriken

Wie hoch ist die Prävalenz von „sepsis“ (Anteil Positiv)?

Baue das DUMMSTE Modell: Es sagt immer „keine Sepsis“ vorher. Berechne Genauigkeit und Sensitivität.

Was ist die Lehre für alle Screening-/Frühwarn-Modelle der Medizin? (1 Satz.)
- Auf hohe Sensitivität tunen, Falsch-positive in Kauf nehmen, um keine Krankheitsfälle zu verpassen.

Notiert: Prävalenz, Genauigkeit & Sensitivität des Dummy-Modells + 1 Satz, warum Genauigkeit hier lügt.
- 97% Genauigkeit liegt hier nur vor, da der Großteil der Daten auf die Pauschalaussage des Dummymodells passt.

In [10]:
print(f"Prävalenz: {df['sepsis'].mean():.1%}")

Prävalenz: 3.0%


In [11]:
pred = 0 
genauigkeit = (df["sepsis"] == pred).mean()
sens = 0.0 # kein einziger Positiver erkannt!
print(f"Genauigkeit: {genauigkeit:.1%} Sensitivität: {sens:.0%}")

Genauigkeit: 97.0% Sensitivität: 0%


### Aufgabe 5 (Bonus) —Die Datenleakage-Falle

Mache einen 80/20-Split auf ZEILEN-Ebene (train_test_split ohne groups).
Prüfe: Kommen dieselben patient_id in Training UND Test vor? Wie viele?

Mache die Aufteilung korrekt (GroupShuffleSplit) —wie ändert sich die Anzahl der Überschneidungen?

Erwartung: Zeilen-Split → mehrere Überschneidungen · Patient-Split → genau 0. Warum ist das so?

In [12]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit

In [13]:
tr, te = train_test_split(
    df.index, 
    test_size=0.2, 
    random_state=42
    )

print("Überschneidung (falsch):",
len(set(df.loc[tr,"patient_id"]) & set(df.loc[te,"patient_id"])))

# df beinhaltet sowieso nur noch eindeutige patient_id, daher ist die Überschneidung 0

Überschneidung (falsch): 0


In [14]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(df, groups=df["patient_id"]))
print("Überschneidung (richtig):",
len(set(df.loc[tr,"patient_id"]) & set(df.loc[te,"patient_id"])))

Überschneidung (richtig): 0


In [16]:
# Bonus-Check: Subgruppen (Vorlesung „Subgruppen ab Tag 1“)
df.groupby("sex")["sepsis"].mean()

sex
M    0.020408
W    0.039216
Name: sepsis, dtype: float64

In [17]:
df.groupby("admission_day")["temp"].mean()

admission_day
0     37.355556
1     37.195455
2     37.185714
3     37.455000
4     37.158333
5     37.256250
6     37.250000
7     37.040000
8     37.668750
9     37.110526
10    37.000000
11    37.142857
12    36.926667
13    36.965000
14    37.030000
15    37.150000
16    36.772727
17    36.735714
18    36.995652
19    37.000000
20    36.847059
21    37.066667
22    37.042857
23    37.064286
24    37.190000
25    37.075000
26    37.150000
27    37.206667
28    37.275000
29    37.216667
30    37.175000
Name: temp, dtype: float64

### Zusammenfassung & nächste Woche

- Ihre Checkliste: Struktur → fehlende Werte (Muster!) → Duplikate/EMPI → Plausibilität → Ungleichgewicht →
Aufteilung auf Patientenebene.
- Alles dokumentieren: Welche Strategie, warum —das ist Datenmanagement, keine Magie.
- Nächste Woche (3): Klassifikation I —kNN & logistische Regression (Übung: erstes eigenes Modell + Metriken).
- Mitnahme W3: saubere explorative Datenanalyse + Merkmale + patientenweise Aufteilung = eure Datenbasis für das
erste Modell.